# ALS Collaborative Filtering - EDA
### Based on: Padhy, N. et al. (2024) - A Recommendation System for E-Commerce Products Using Collaborative Filtering Approaches
### Journal: Engineering Proceedings (MDPI), Vol. 67, No. 1, Article 50
### DOI: https://doi.org/10.3390/engproc2024067050

In [0]:
from pyspark.sql.functions import col, count, countDistinct, when
from pyspark.sql.functions import max as spark_max, sum as spark_sum, avg
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

COLOR_MAIN = "#1B3A6B"
COLOR_WARN = "#E65C00"
COLOR_OK = "#00695C"
COLOR_BAD = "#B71C1C"
COLOR_NEUTRAL = "#9E9E9E"

print("imports done")

In [0]:
orders_df = spark.table("bharatmart.silver.slv_orders")
cart_df = spark.table("bharatmart.silver.slv_cart_events")
reviews_df = spark.table("bharatmart.silver.slv_reviews")
products_df = spark.table("bharatmart.silver.slv_products")

print("tables loaded")

In [0]:
print(f"orders : {orders_df.count():>12,}")
print(f"cart_events : {cart_df.count():>12,}")
print(f"reviews : {reviews_df.count():>12,}")
print(f"products : {products_df.count():>12,}")

**check orders schema**

In [0]:
orders_df.printSchema()

**check cart schema**

In [0]:
cart_df.printSchema()

**check reviews schema**

In [0]:
reviews_df.printSchema()

three tables loaded. all three have customer_id and product_id which is what
ALS needs. 

orders is the cleanest - product_id is a direct column, no parsing needed.

cart_events has an action column, not event_type as initially expected. 
need to check what values action actually contains before filtering for 
add_to_cart events. if the column has different values we adjust accordingly.

reviews has a _is_ghost_order flag from the silver layer. ghost reviews 
are linked to orders with no customer_id attached - they cannot be used 
as a signal. will filter these before building the interaction matrix.

**check null on key columns across all three tables**

In [0]:
print("--- orders ---")
print(f"customer_id null : {orders_df.filter(col('customer_id').isNull()).count():,}")
print(f"product_id null  : {orders_df.filter(col('product_id').isNull()).count():,}")

print("--- cart_events ---")
print(f"customer_id null : {cart_df.filter(col('customer_id').isNull()).count():,}")
print(f"product_id null  : {cart_df.filter(col('product_id').isNull()).count():,}")

print("--- reviews ---")
print(f"customer_id null : {reviews_df.filter(col('customer_id').isNull()).count():,}")
print(f"product_id null  : {reviews_df.filter(col('product_id').isNull()).count():,}")

**check distinct action values in cart_events**

In [0]:
cart_df.groupBy("action").count().orderBy(col("count").desc()).show()

cart_events is clean - zero nulls on both key columns. good.

reviews is also clean. the _is_ghost_order flag in silver already handled 
the problematic rows at that layer.

orders has two null problems.

32,284 null customer_id - these are the same ghost orders model 1 found 
and dropped. expected, not a surprise.

55,000 null product_id is unexpected. that is 3.3% of all orders with no 
product attached. these cannot be used in ALS - you cannot recommend a 
product that has no id. need to check what order_status these rows have 
before dropping them. if they are all cancelled or failed orders the drop 
is clean. if some are delivered orders something is wrong upstream.

cart action column has different names than expected. doc assumed 
add_to_cart but the actual values are cart_add, cart_view, cart_remove, 
wishlist_add, and a few others. cart_add is the primary signal at 1.8M rows.
cart_remove is a negative signal - customer decided against the product. 
will exclude it from the interaction matrix. wishlist_add at 458K is an 
intent signal worth including at lower confidence weight.

**Investigate null product_id orders**

In [0]:
null_prod = orders_df.filter(col("product_id").isNull())

null_prod.groupBy("order_status").count() \
    .orderBy(col("count").desc()).show()

**check if null product_id orders have customer_id**

In [0]:
print(f"null product_id, has customer_id  : {null_prod.filter(col('customer_id').isNotNull()).count():,}")
print(f"null product_id, null customer_id : {null_prod.filter(col('customer_id').isNull()).count():,}")

55,000 null product_id orders all have a valid customer_id. every single one.
so these are real customers who placed real orders - the customer side is 
intact but the product side never got attached.

the order status breakdown makes this worse than expected. 8,308 are 
delivered. 13,793 are shipped. 13,695 are confirmed. these are not failed 
or test orders - they went through the full fulfillment pipeline with no 
product recorded.

this is an upstream data quality issue. something in the order creation or 
item attachment process is dropping product_id for roughly 1 in 30 orders. 
it cannot be fixed at the ML layer.

for ALS these rows are unusable. you cannot learn a user-item preference 
when the item half is missing. they will be dropped before building the 
interaction matrix - but this is worth flagging as a known data gap. 
those 8,308 delivered orders represent real purchase signals we are losing.

after dropping nulls on both customer_id and product_id, clean orders will 
be 1,661,695 - 32,284 - 55,000 = 1,574,411 rows. that is still a very 
strong interaction signal for ALS.

**clean orders and confirm count**

In [0]:
clean_orders = orders_df.filter(
    col("customer_id").isNotNull() & col("product_id").isNotNull()
)

print(f"original orders : {orders_df.count():,}")
print(f"clean orders : {clean_orders.count():,}")
print(f"dropped : {orders_df.count() - clean_orders.count():,}")

**display sample of clean orders**

In [0]:
display(clean_orders.select(
    "customer_id", "product_id", "order_status", "order_date"
).limit(5))

dropped 87,284 rows total - 32,284 ghost orders with no customer and 
55,000 orders with no product attached. the two problems don't overlap 
at all - no row had both nulls.

1,574,411 clean orders remain. customer_id and product_id are both 
string UUIDs - CUST prefix for customers, PROD prefix for products.

ALS in PySpark MLlib requires integer IDs, not strings. StringIndexer 
will map these UUIDs to sequential integers before training. the mapping 
gets saved to MLflow so batch scoring can reverse it back to the original 
UUIDs in the output table.

the sample looks correct - real customers, real products, real order 
statuses. this is the base table for building the interaction matrix.

**how many unique customers and products in clean orders**

In [0]:
n_customers = clean_orders.select("customer_id").distinct().count()
n_products  = clean_orders.select("product_id").distinct().count()

print(f"unique customers : {n_customers:,}")
print(f"unique products : {n_products:,}")
print(f"possible pairs : {n_customers * n_products:,}")

**orders per customer distribution**

In [0]:
orders_per_cust = clean_orders.groupBy("customer_id") \
    .agg(count("order_id").alias("order_count"))

orders_per_cust_pdf = orders_per_cust.select("order_count").toPandas()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(orders_per_cust_pdf["order_count"], bins=50,
        color=COLOR_MAIN, edgecolor="white")
ax.set_xlabel("Orders per Customer")
ax.set_ylabel("Customer Count")
ax.set_title("Orders per Customer Distribution")
plt.tight_layout()
plt.show()

91,634 unique customers. 47,030 unique products. 4.3 billion possible 
user-item pairs. only 1,574,411 are actually observed that's 0.037% 
of the matrix filled in. 99.963% sparse.

this is normal for e-commerce. most customers buy a small fraction of 
the catalog. the whole point of ALS is to learn latent factors that let 
it make confident recommendations even in the empty 99.9% of the matrix.

Padhy et al. (2024) worked with a similarly sparse rating matrix of 
1,048,100 records and noted that ALS handles missing values effectively 
precisely because it fills them in through matrix factorisation rather 
than relying on observed data alone.

the orders per customer chart has the same sharp spike at 19 orders we 
saw in model 2 nearly 5,700 customers at exactly that value. a natural 
distribution does not produce spikes this sharp at one specific number. 
it appeared in recency analysis and now in purchase frequency. it's a 
data generation pattern in the BharatMart simulation, not real customer 
behaviour. it does not block ALS but it's worth knowing it's there.

the distribution runs from 1 order up to about 54, with most customers 
sitting between 5 and 25 orders. that is a healthy signal range for ALS enough interactions per customer to learn preferences without being 
dominated by a handful of super-buyers.

**compute sparsity properly**

In [0]:
actual_interactions = clean_orders.count()
possible_pairs = 91634 * 47030

sparsity = 1 - (actual_interactions / possible_pairs)

print(f"actual interactions : {actual_interactions:,}")
print(f"possible pairs : {possible_pairs:,}")
print(f"sparsity : {sparsity*100:.4f}%")

**cold start: customers with fewer than 3 orders**

In [0]:
cold_start = orders_per_cust.filter(col("order_count") < 3)
warm_users  = orders_per_cust.filter(col("order_count") >= 3)

print(f"cold start customers (<3 orders) : {cold_start.count():,}")
print(f"warm customers (>=3 orders) : {warm_users.count():,}")
print(f"cold start % : {cold_start.count()/orders_per_cust.count()*100:.1f}%")

2,692 customers have fewer than 3 orders - 2.9% of the base. that is a 
very low cold start rate. most e-commerce datasets have 10-20% cold start 
customers. BharatMart's customer base is well-established enough that 
almost everyone has left a usable signal.

these 2,692 customers will be excluded from ALS training. with only 1 or 
2 orders there is not enough signal to learn meaningful latent factors. 
the model would be guessing. they get popularity-based fallback 
recommendations instead, top products in their RFM segment's most 
common category. the rec_source column in the output table marks them 
as popularity_fallback so downstream users know the source.

88,942 warm customers go into ALS. that is the actual training population.

sparsity confirmed at 99.96%. this is expected and not a problem ALS 
was designed for exactly this kind of matrix. Padhy et al. (2024) note 
that ALS is suited for parallel computation and handling huge sparse 
datasets, which is why PySpark MLlib's ALS implementation is the right 
tool here over SVD or KNNBasic which struggle at this scale.

**orders per product distribution**

In [0]:
orders_per_prod = clean_orders.groupBy("product_id") \
    .agg(count("order_id").alias("order_count"))

orders_per_prod_pdf = orders_per_prod.select("order_count").toPandas()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(orders_per_prod_pdf["order_count"], bins=50,
        color=COLOR_OK, edgecolor="white")
ax.set_xlabel("Orders per Product")
ax.set_ylabel("Product Count")
ax.set_title("Orders per Product Distribution")
plt.tight_layout()
plt.show()

**products with very few orders**

In [0]:
cold_items = orders_per_prod.filter(col("order_count") < 5)
warm_items  = orders_per_prod.filter(col("order_count") >= 5)

print(f"products with <5 orders  : {cold_items.count():,}")
print(f"products with >=5 orders : {warm_items.count():,}")
print(f"cold item % : {cold_items.count()/orders_per_prod.count()*100:.1f}%")

only 345 products have fewer than 5 orders 0.7% of the catalog. 
that is exceptionally low. almost every product in BharatMart's 50,000 
item catalog has been purchased enough times to generate a usable signal 
for ALS item factors.

the distribution shape is worth paying attention to. there are two humps, one peak around 10-13 orders and a second larger peak around 38-45 
orders. a natural catalog would produce a smooth right-skewed curve with 
one peak near the low end. two humps suggest two distinct product 
populations in the data likely a split between newer or niche products 
that get occasional purchases and established catalog items that sell 
consistently. this is a simulation data pattern but it does not affect 
ALS. the algorithm learns from whatever distribution exists.

the range goes out to about 80 orders max. no single product dominates 
the catalog which is good, if a handful of products had 10,000+ orders 
each they would pull ALS latent factors toward them and make every 
customer look like they want the same things.

cold item rate of 0.7% is not a concern. we will not apply a separate 
item cold start filter items with fewer than 5 orders stay in the 
matrix and ALS will simply learn weaker factors for them.

**check cart action values we will use**

In [0]:
cart_df.groupBy("action") \
    .count() \
    .orderBy(col("count").desc()) \
    .show()

**check nulls in cart for our key columns**

In [0]:
print(f"cart customer_id null : {cart_df.filter(col('customer_id').isNull()).count()}")
print(f"cart product_id null  : {cart_df.filter(col('product_id').isNull()).count()}")

cart_signals = cart_df.filter(col("action").isin("cart_add", "wishlist_add"))
print(f"cart_add + wishlist_add rows : {cart_signals.count():,}")

cart_events is clean - zero nulls on customer_id and product_id across 
all 4.77M rows.

for the interaction matrix we use two action types.

cart_add at 1,832,289 rows is the primary intent signal. a customer 
added a product to their cart, they were seriously considering it. 
this gets a confidence weight of 0.5 in the interaction matrix since 
it did not convert to a purchase.

wishlist_add at 458,698 rows is a secondary intent signal. weaker than 
cart_add but still a real preference signal, the customer wanted to 
remember the product. same 0.5 weight as cart_add.

cart_view at 1.6M rows is too weak browsing is not intent. cart_remove 
and remove are negative signals, the customer actively rejected the 
product. these two are excluded from the interaction matrix entirely. 
including negative signals in ALS implicit feedback would corrupt the 
confidence weights.

the smaller actions add, update_qty, move_to_wishlist, save_for_later

**review coverage and rating distribution**

In [0]:
review_coverage = reviews_df.filter(
    col("_is_ghost_order") == False
).count()

print(f"total reviews : {reviews_df.count():,}")
print(f"non-ghost reviews : {review_coverage:,}")
print(f"ghost reviews : {reviews_df.count() - review_coverage:,}")
print(f"review coverage vs orders : {review_coverage/clean_orders.count()*100:.1f}%")

**rating distribution**

In [0]:
clean_reviews = reviews_df.filter(col("_is_ghost_order") == False)

rating_pdf = clean_reviews.groupBy("rating") \
    .count() \
    .orderBy("rating") \
    .toPandas()

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(rating_pdf["rating"].astype(str), rating_pdf["count"],
       color=COLOR_MAIN, edgecolor="white", width=0.5)
ax.set_xlabel("Rating")
ax.set_ylabel("Review Count")
ax.set_title("Rating Distribution in Reviews")
plt.tight_layout()
plt.show()

12,280 ghost reviews dropped these were linked to orders with no real 
customer. 398,497 clean reviews remain.

review coverage is 25.3% one in four orders has a review. the doc 
estimated 18% so actual coverage is better than expected. still, 74.7% 
of orders have no review which confirms that explicit ratings alone 
cannot drive the recommendation engine. implicit feedback from purchases 
and cart events is the right foundation.

the rating distribution is a J-curve. 5-star reviews dominate at ~155,000. 
4-star is second at ~98,000. then there is a sharp drop 3-star at 
~58,000, 1-star at ~51,000, and 2-star lowest at ~

**check how many reviews are high vs low rating**

In [0]:
print("rating breakdown:")
clean_reviews.groupBy("rating") \
    .count() \
    .orderBy("rating") \
    .show()

high_rating = clean_reviews.filter(col("rating") >= 4).count()
low_rating  = clean_reviews.filter(col("rating") <= 2).count()

print(f"high ratings (4-5) to include : {high_rating:,}")
print(f"low ratings (1-2) to exclude : {low_rating:,}")
print(f"neutral (3) — borderline: {clean_reviews.filter(col('rating') == 3).count():,}")

**top 10 most purchased products**

In [0]:
top_products = clean_orders.groupBy("product_id") \
    .agg(
        count("order_id").alias("purchase_count"),
        countDistinct("customer_id").alias("unique_buyers")
    ) \
    .orderBy(col("purchase_count").desc())

display(top_products.limit(10))

the top product has 88 purchases from 88 unique buyers. purchase_count 
equals unique_buyers for almost every product in the top 10 meaning 
customers are buying each product at most once. repeat purchases of the 
same product are extremely rare in this catalog.

this has a direct implication for ALS. the interaction matrix will be 
almost entirely binary "0 or 1" purchases per user-item pair. the 
confidence weight from purchase frequency (purchase_count * 1.0) will 
be 1.0 for nearly every interaction. there is no meaningful signal from 
repeat purchase frequency in this dataset.

the max of 88 purchases for the most popular product also confirms 
there are no dominant bestsellers pulling the catalog. with 47,030 
products and 1.57M orders the average is about 33 orders per product. 
the top product at 88 is only 2.7x the average. this is a very flat 
popularity distribution, good for ALS because it means the model 
cannot just recommend the same 10 bestsellers to everyone and look 
accurate.

Padhy et al. (2024) note that ALS is suited for exactly this kind of 
sparse, flat interaction matrix where explicit ratings or repeat 
purchases are not available. the latent factors ALS learns will capture 
product category affinity and co-purchase patterns rather than frequency.

**confirm how many user-item pairs have repeat purchases**

In [0]:
repeat_purchases = clean_orders.groupBy("customer_id", "product_id") \
    .agg(count("order_id").alias("times_purchased")) \
    .filter(col("times_purchased") > 1)

print(f"user-item pairs with repeat purchase : {repeat_purchases.count():,}")
print(f"total unique user-item pairs : {clean_orders.select('customer_id','product_id').distinct().count():,}")

**signal source summary: how many unique user-item pairs from each source**

In [0]:
purchase_pairs = clean_orders.select("customer_id", "product_id").distinct().count()

cart_pairs = cart_df.filter(col("action").isin("cart_add", "wishlist_add")) \
    .select("customer_id", "product_id").distinct().count()

review_pairs = clean_reviews.filter(col("rating") >= 4) \
    .select("customer_id", "product_id").distinct().count()

print(f"unique pairs from purchases : {purchase_pairs:,}")
print(f"unique pairs from cart signals : {cart_pairs:,}")
print(f"unique pairs from high reviews : {review_pairs:,}")

only 428 user-item pairs have a repeat purchase out of 1,573,983 total 
pairs. that is 0.03%. the interaction matrix is binary in practice 
almost every customer bought each product exactly once or not at all.

this means the confidence weight formula purchase_count * 1.0 collapses 
to just 1.0 for 99.97% of interactions. there is no meaningful frequency 
signal to exploit in purchases alone.

this actually makes the cart and review signals more important, not less. 
they are the only source of differentiation in confidence weights.

three signal sources going into the interaction matrix:
- purchases: 1,573,983 unique pairs at confidence 1.0
- cart signals (cart_add + wishlist_add): 1,246,664 unique pairs at confidence 0.5
- high reviews (rating 4-5): 253,619 unique pairs at confidence 2.0

when the same user-item pair appears in multiple sources we take the max 
confidence. a product someone both purchased and gave 5 stars gets 
confidence 2.0, not 3.0. this prevents a single enthusiastic customer 
from dominating the model's item factors.

total unique signal pairs before deduplication spans roughly 3M 
combinations. after taking max per pair the final interaction matrix 
will be smaller — the next step checks the actual merged size.

**build merged interaction matrix and check final size**

In [0]:
from pyspark.sql.functions import lit, spark_partition_id
from pyspark.sql.functions import greatest

purchase_df = clean_orders.select(
    col("customer_id"),
    col("product_id"),
    lit(1.0).alias("confidence")
)

cart_signal_df = cart_df.filter(col("action").isin("cart_add", "wishlist_add")).select(
    col("customer_id"),
    col("product_id"),
    lit(0.5).alias("confidence")
)

review_signal_df = clean_reviews.filter(col("rating") >= 4).select(
    col("customer_id"),
    col("product_id"),
    lit(2.0).alias("confidence")
)

all_signals = purchase_df.union(cart_signal_df).union(review_signal_df)

interaction_df = all_signals.groupBy("customer_id", "product_id") \
    .agg(spark_max("confidence").alias("confidence"))

print(f"total unique user-item pairs in matrix : {interaction_df.count():,}")

**confidence weight distribution**

In [0]:
conf_dist = interaction_df.groupBy("confidence") \
    .count() \
    .orderBy("confidence") \
    .toPandas()

print(conf_dist.to_string(index=False))

final interaction matrix: 1,600,485 unique user-item pairs.

confidence breakdown:
- 2.0: 253,619 pairs - purchased and rated 4 or 5 stars
- 1.0: 1,325,892 pairs - purchased, no high review
- 0.5: 20,974 pairs - cart or wishlist add only, never purchased

the 0.5 group is much smaller than expected. cart signals contributed 
1,246,664 pairs but only 20,974 survived as pure cart-only interactions. 
the rest over 1.2M  overlapped with purchases and got absorbed into 
the 1.0 bucket since we took max confidence per pair. 

this tells us something real about BharatMart customers. when they add 
something to cart they usually buy it. cart abandonment without purchase 
is rare. the cart is not a wishlist, it is a checkout waiting room.

the 2.0 bucket at 253,619 is the most valuable slice of the matrix. 
these are customers who not only bought a product but cared enough to 
leave a positive review. ALS will learn the strongest item factor signals 
from this group.

the matrix is clean and ready for StringIndexer and ALS training. 
Padhy et al. (2024) explicitly warn that ALS has overfitting risk without 
proper regularisation — the regParam tuning in notebook 11b directly 
addresses this finding. with 1.6M interactions across 88,942 warm 
customers and 47,030 products the matrix is large enough that 
regularisation is critical to prevent the model memorising training 
interactions instead of generalising.

**check 6 month date range for interaction window**

In [0]:
from pyspark.sql.functions import months_between, current_date, min as spark_min

date_stats = clean_orders.select(
    spark_min("order_date").alias("earliest"),
    spark_max("order_date").alias("latest")
).toPandas()

print(f"earliest order : {date_stats['earliest'][0]}")
print(f"latest order : {date_stats['latest'][0]}")

**how many orders fall in last 6 months**

In [0]:
from pyspark.sql.functions import months_between, current_date

recent_orders = clean_orders.filter(
    months_between(current_date(), col("order_date")) <= 6
)

print(f"orders in last 6 months : {recent_orders.count():,}")
print(f"total clean orders : {clean_orders.count():,}")
print(f"6-month coverage : {recent_orders.count()/clean_orders.count()*100:.1f}%")

orders span from January 2020 to March 2026 over 6 years of history.

the 6-month window (last 6 months) captures only 240,762 orders - 15.3% 
of the total. that means 85% of purchase history gets cut.

the reasoning for a 6-month window is sound - a product someone bought 
in 2020 tells us less about what they want today than something they 
bought last month. customer preferences shift. new products replace old 
ones. a 2020 purchase of a phone case is irrelevant if the customer has 
a different phone now.

but cutting 85% of data is a real tradeoff. if too many customers drop 
out of the 6-month window the cold start problem gets worse and ALS has 
less signal to learn from. need to check how many unique customers 
survive before deciding whether 6 months is the right cutoff or whether 
we need to extend to 12 months.

**unique customers in 6 month window vs full history**

In [0]:
recent_customers = recent_orders.select("customer_id").distinct().count()
all_customers = clean_orders.select("customer_id").distinct().count()

recent_products = recent_orders.select("product_id").distinct().count()
all_products = clean_orders.select("product_id").distinct().count()

print(f"customers in 6-month window : {recent_customers:,}  ({recent_customers/all_customers*100:.1f}% of total)")
print(f"customers in full history : {all_customers:,}")
print()
print(f"products in 6-month window  : {recent_products:,}  ({recent_products/all_products*100:.1f}% of catalog)")
print(f"products in full history : {all_products:,}")

**try 12 months and compare**

In [0]:
window_12m = clean_orders.filter(
    months_between(current_date(), col("order_date")) <= 12
)

customers_12m = window_12m.select("customer_id").distinct().count()
orders_12m = window_12m.count()

print(f"12-month orders : {orders_12m:,}  ({orders_12m/clean_orders.count()*100:.1f}%)")
print(f"12-month customers : {customers_12m:,}  ({customers_12m/all_customers*100:.1f}%)")

6-month window keeps 84,840 customers - 92.6% of the total base. only 
6,794 customers become invisible at 6 months. these are customers who 
bought something more than 6 months ago and never came back - which 
means model 1 already flagged most of them as churned. recommending 
products to customers who haven't engaged in over 6 months is low 
priority anyway.

12 months recovers 5,125 more customers for a total of 89,965  only 
a 5.6% improvement over 6 months, but it doubles the data window from 
240K to 408K orders and pulls in older, less relevant purchase history.

the tradeoff does not justify the extension. 6 months is the right 
cutoff for two reasons. first, customer preferences in e-commerce shift 
faster than 6 months - a product category someone bought in early 2025 
may no longer reflect what they want today. second, the 7.4% of customers 
lost are largely churned customers already identified by model 1 - their 
old purchase history would add noise, not signal.

6-month window is confirmed. 84,840 customers, 43,138 products, 
240,762 orders going into the interaction matrix.

**final interaction matrix size with 6-month window**

In [0]:
recent_purchases = clean_orders.filter(
    months_between(current_date(), col("order_date")) <= 6
)

recent_cart = cart_df.filter(
    col("action").isin("cart_add", "wishlist_add")
).filter(
    months_between(current_date(), col("event_date")) <= 6
)

recent_reviews = clean_reviews.filter(col("rating") >= 4).filter(
    months_between(current_date(), col("review_date")) <= 6
)

print(f"6-month purchases : {recent_purchases.count():,}")
print(f"6-month cart : {recent_cart.count():,}")
print(f"6-month reviews : {recent_reviews.count():,}")

**build 6-month interaction matrix and check final pair count**

In [0]:
p_df = recent_purchases.select(
    col("customer_id"), col("product_id"), lit(1.0).alias("confidence")
)

c_df = recent_cart.select(
    col("customer_id"), col("product_id"), lit(0.5).alias("confidence")
)

r_df = recent_reviews.select(
    col("customer_id"), col("product_id"), lit(2.0).alias("confidence")
)

interaction_6m = p_df.union(c_df).union(r_df) \
    .groupBy("customer_id", "product_id") \
    .agg(spark_max("confidence").alias("confidence"))

print(f"final 6-month interaction pairs : {interaction_6m.count():,}")

final 6-month interaction matrix: 247,711 unique user-item pairs.

raw signals before deduplication:
- purchases : 240,762
- cart       : 384,479
- reviews    : 41,897
- total raw  : 667,138

after taking max confidence per pair only 247,711 unique pairs survive. 
419,427 signals collapsed into existing pairs — mostly cart events 
overlapping with purchases from the same customer on the same product.

cart events (384,479) outnumber purchases (240,762) in the 6-month 
window. this is the opposite of the full history pattern where purchases 
dominated. it means recent customers are browsing more than converting, 
a normal pattern in the most recent months where some orders are still 
in processing or shipped status and not yet reflecting final purchase 
behaviour.

the bigger concern is signal density. 247,711 pairs across 84,840 
customers gives an average of 2.9 interactions per customer. that is 
very thin for ALS the model needs enough interactions per user to 
learn meaningful latent factors. the full history gave 17.4 interactions 
per customer on average.

this is a real tradeoff that needs a decision before training. the 
6-month window preserves recency but sacrifices density. the full 
history gives 6.5x more signal per customer but includes purchase 
behaviour from 2020-2022 that may no longer reflect current preferences.

a hybrid approach is worth considering use full history for customers 
with fewer than 10 interactions in the last 6 months, and 6-month only 
for customers with sufficient recent signal. this preserves recency 
where data is rich and falls back to full history where it is thin.

**check average interactions per customer in 6-month matrix**

In [0]:
interactions_per_cust = interaction_6m.groupBy("customer_id") \
    .agg(count("product_id").alias("interaction_count"))

stats = interactions_per_cust.select(
    avg("interaction_count").alias("mean"),
    spark_max("interaction_count").alias("max")
).toPandas()

low_signal = interactions_per_cust.filter(col("interaction_count") < 5).count()
total = interactions_per_cust.count()

print(f"avg interactions per customer : {stats['mean'][0]:.1f}")
print(f"max interactions per customer : {stats['max'][0]}")
print(f"customers with <5 interactions : {low_signal:,}  ({low_signal/total*100:.1f}%)")

**same check on full history matrix**

In [0]:
interactions_per_cust_full = interaction_df.groupBy("customer_id") \
    .agg(count("product_id").alias("interaction_count"))

stats_full = interactions_per_cust_full.select(
    avg("interaction_count").alias("mean"),
    spark_max("interaction_count").alias("max")
).toPandas()

low_signal_full = interactions_per_cust_full.filter(col("interaction_count") < 5).count()
total_full = interactions_per_cust_full.count()

print(f"avg interactions per customer : {stats_full['mean'][0]:.1f}")
print(f"max interactions per customer : {stats_full['max'][0]}")
print(f"customers with <5 interactions : {low_signal_full:,}  ({low_signal_full/total_full*100:.1f}%)")

6-month window fails on signal density. 85.1% of customers 72,606 
out of 84,840 have fewer than 5 interactions in the last 6 months. 
average is 2.9 interactions per customer with a max of only 12. ALS 
needs enough user-item pairs to learn what a customer actually prefers 2-3 interactions is guessing, not learning.

full history tells a completely different story. average 17.5 
interactions per customer, max 55, and only 8.6% of customers fall 
below 5 interactions. the model has real signal to work with.

the concern about stale data from 2020-2022 is valid but secondary. 
ALS with implicitPrefs = True uses confidence weights, the 2.0 weight 
on recent high-rated purchases naturally amplifies recent signal without 
discarding older purchase history entirely. recent behaviour gets more 
weight through the review signal. old behaviour provides the density 
the model needs to learn latent factors.

full history is the training dataset for ALS. the 6-month recency 
concern is handled through confidence weights, not data cutoff.

final training matrix: 1,600,485 unique user-item pairs, 91,634 
customers, 47,030 products, average 17.5 interactions per customer.

**interactions per customer histogram, full history**

In [0]:
interactions_pdf = interactions_per_cust_full.select("interaction_count").toPandas()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(interactions_pdf["interaction_count"], bins=50,
        color=COLOR_MAIN, edgecolor="white")
ax.set_xlabel("Interactions per Customer")
ax.set_ylabel("Customer Count")
ax.set_title("Interactions per Customer — Full History (ALS Training Matrix)")
plt.tight_layout()
plt.show()

the distribution runs from 1 to about 55 interactions per customer. 
the shape is right-skewed which is exactly what we want most customers 
have a moderate number of interactions and a smaller group has many.

two sharp spikes at 15 and 29 interactions. we've seen this pattern 
three times now 19 orders in the customer frequency chart, 19 in the 
product chart, and now 15 and 29 here. it's a simulation data artefact, 
not real customer behaviour. it does not affect ALS the model learns 
from the full distribution, not individual count values.

the important thing is the range. most customers sit between 5 and 30 
interactions. that is enough for ALS to learn meaningful latent factors. 
the model needs at least 5 interactions to place a user reliably in 
latent space at 8.6% below that threshold we have strong coverage.

EDA is complete. the interaction matrix is ready for notebook 11b.

## EDA Summary

data sources: slv_orders, slv_cart_events, slv_reviews

key findings:

orders - dropped 87,284 rows. 32,284 ghost orders with no customer_id 
and 55,000 orders with no product_id (3.3% of orders, includes 8,308 
delivered orders - upstream data quality issue, flagged but not fixable 
at ML layer). clean orders: 1,574,411.

interaction matrix - 1,600,485 unique user-item pairs. three confidence 
levels: 2.0 for purchase + high review, 1.0 for purchase only, 0.5 for 
cart/wishlist only. 99.96% sparse. 428 repeat purchases (0.03%) matrix 
is effectively binary, frequency scaling adds nothing.

cold start - 2,692 customers with fewer than 3 orders (2.9%). these get 
popularity fallback recommendations. 88,942 warm customers go into ALS.

cart signals - action column is cart_add and wishlist_add, not event_type 
as originally documented. cart_remove and cart_view excluded. 

reviews - 12,280 ghost reviews dropped. 398,497 clean. 25.3% coverage. 
ratings 1-2 excluded from interaction matrix (86,559 rows). rating 3 
excluded borderline signal adds noise. only 4-5 star reviews used.

time window - 6-month window rejected. 85.1% of customers had fewer 
than 5 interactions in 6 months — too thin for ALS. full history used. 
confidence weights handle recency, not data cutoff.

ready for notebook 11b - StringIndexer, ALS training, hyperparameter 
tuning, MLflow registration.